# PyTorch & Einops: 60-Minute Tensor Fundamentals

**For ARENA and mechanistic interpretability.** Work through the cells in order. **Predict shapes before running code.** Exercises have separate hidden-in-plain-sight solution sections you can skip until ready.

| Section | Time |
|---|---:|
| Setup | 2 min |
| Shapes | 8 min |
| Indexing | 10 min |
| Broadcasting | 12 min |
| Reshape / stack / cat | 8 min |
| Einops | 10 min |
| Final challenge | 10 min |

If `einops` is missing, run `%pip install einops` in a notebook cell.

## 0 · Setup (2 min)

In [ ]:
import torch as t
from einops import rearrange, repeat, reduce
t.set_printoptions(precision=2)
print("PyTorch", t.__version__)

## 1 · Shapes (8 min)

A tensor is a multidimensional array. The **meaning** of each dimension comes from the problem: `(batch, seq, d_model)` is a common convention, not something PyTorch knows intrinsically.

- A scalar has shape `()`.
- A vector has shape `(n,)`.
- A matrix has shape `(rows, columns)`.
- A batched matrix has shape `(batch, rows, columns)`.

In [ ]:
scalar = t.tensor(5)
vector = t.tensor([1, 2, 3])
matrix = t.tensor([[1, 2, 3], [4, 5, 6]])
batch = t.randn(4, 2, 3)
for name, x in [("scalar",scalar),("vector",vector),("matrix",matrix),("batch",batch)]:
    print(name, x.shape, "ndim:", x.ndim)

### Exercise 1 — predict without running

Given `A=(5,10)`, `B=(3,5,10)`, `C=(2,3,4,5)`, predict `A.ndim`, `B.shape[-1]`, `C.shape[1:]`, `C.numel()`. If `B` represents 3 images, which axis is the image axis?

In [ ]:
A = t.randn(5,10); B = t.randn(3,5,10); C = t.randn(2,3,4,5)
# Write predictions here as comments, then uncomment to check:
# print(A.ndim, B.shape[-1], C.shape[1:], C.numel())

**Solution 1:** `2`, `10`, `(3, 4, 5)`, `120`; images are axis 0.

## 2 · Indexing & slicing (10 min)

A single integer index **removes** an axis; a slice (`:` or `0:1`) **keeps** it. `...` fills in unspecified axes. `None` / `unsqueeze` inserts a size-one axis.

In [ ]:
x = t.arange(24).reshape(2,3,4)
print(x)
for label, y in [("x[0]",x[0]),("x[:,0]",x[:,0]),("x[:,0:1]",x[:,0:1]),("x[...,0]",x[...,0]),("x[..., :2]",x[...,:2]),("x[:,None,:,:]",x[:,None,:,:])]:
    print(label, tuple(y.shape))

### Exercise 2

Let `x` have shape `(8,12,64)` = `(batch, seq, embedding)`.

Predict the shapes of `x[0]`, `x[:,0]`, `x[:,:3]`, `x[...,-1]`, `x[:,None,:,:]`, `x[:,:,None,:]`, `x[0,:,:10]`.

Then write expressions to (a) select first token embeddings for all batches, (b) select first 5 features for every token, (c) insert an axis between sequence and embedding.

In [ ]:
x = t.randn(8,12,64)
# YOUR WORK
# a = ...
# b = ...
# c = ...

**Solution 2:** shapes `(12,64)`, `(8,64)`, `(8,3,64)`, `(8,12)`, `(8,1,12,64)`, `(8,12,1,64)`, `(12,10)`. Expressions: `x[:,0,:]`, `x[...,:5]`, `x[:,:,None,:]`.

## 3 · Broadcasting (12 min)

PyTorch aligns axes **from the right**. Each pair of axis lengths must match, or one must be 1 (missing leading axes behave as 1).

```
(5, 1, 3)
(1, 7, 3)
---------
(5, 7, 3)
```

A powerful pattern for all-pairs comparisons: `(n,d)` and `(m,d)` become `(n,1,d)` and `(1,m,d)`, yielding `(n,m,d)`.

In [ ]:
A = t.tensor([[1,2,3],[4,5,6]])
b = t.tensor([10,20,30])
print(A + b)
P = t.randn(5,3); Q = t.randn(7,3)
diff = P[:,None,:] - Q[None,:,:]
print("pairwise differences:", diff.shape)

### Exercise 3

1. Shape of `(4,1,8) + (1,6,8)`?
2. Does `(5,3) + (3,5)` work?
3. Given `points=(10,2)`, compute a `(10,10)` matrix of pairwise Euclidean distances, **without loops**.

In [ ]:
points = t.randn(10,2)
# YOUR WORK
# distances = ...
# assert distances.shape == (10,10)
# assert t.allclose(distances, distances.T, atol=1e-5)

**Solution 3:** `(4,6,8)`; no, 5 and 3 conflict; `distances = t.linalg.vector_norm(points[:,None,:] - points[None,:,:], dim=-1)`.

## 4 · Reshape, stack, cat (8 min)

- `reshape`: changes the shape while preserving element order (may create a copy).
- `stack`: inserts a **new** axis; inputs must have equal shape.
- `cat`: joins along an **existing** axis; other axes must match.
- `unsqueeze`: inserts a size-one axis.

In [ ]:
a = t.tensor([1,2,3]); b = t.tensor([4,5,6])
print("stack dim 0:\n", t.stack((a,b),dim=0))
print("stack dim 1:\n", t.stack((a,b),dim=1))
A = t.randn(2,3); B = t.randn(2,3)
print("cat 0:", t.cat((A,B),dim=0).shape, "cat 1:", t.cat((A,B),dim=1).shape)

### Exercise 4

Given `A` and `B` each shaped `(5,3)`, produce `(2,5,3)`, `(5,2,3)`, `(10,3)`, and `(5,6)` using stack or cat.

In [ ]:
A = t.randn(5,3); B = t.randn(5,3)
# YOUR WORK

**Solution 4:** `t.stack((A,B),0)`, `t.stack((A,B),1)`, `t.cat((A,B),0)`, `t.cat((A,B),1)`.

## 5 · Einops (10 min)

Einops gives **names** to axes. Three essentials:

- `rearrange`: reorder, flatten, or split axes.
- `repeat`: add/repeat axes.
- `reduce`: aggregate across axes that disappear from the output.

`rearrange(x, "b s d -> b d s")` swaps sequence and feature axes. Parentheses group or split axes.

In [ ]:
x = t.randn(32,10,768)
print(rearrange(x,"b s d -> b d s").shape)
print(rearrange(x,"b s d -> b (s d)").shape)
print(rearrange(x,"b s (h dh) -> b h s dh",h=12).shape)
print(repeat(x,"b s d -> b copies s d",copies=3).shape)
print(reduce(x,"b s d -> b d","mean").shape)

### Exercise 5

Given `x=(16,20,128)`, write einops operations to produce:

A. `(20,16,128)`
B. `(16,2560)`
C. `(16,8,20,16)` by splitting embedding into 8 heads
D. `(16,128)` by averaging over tokens
E. `(16,3,20,128)` by repeating sequences three times.

In [ ]:
x = t.randn(16,20,128)
# YOUR WORK

**Solution 5**

```python
rearrange(x, "b s d -> s b d")
rearrange(x, "b s d -> b (s d)")
rearrange(x, "b s (h dh) -> b h s dh", h=8)
reduce(x, "b s d -> b d", "mean")
repeat(x, "b s d -> b copies s d", copies=3)
```

## 6 · Final challenge: pairwise dot products (10 min)

`A=(10,3)` and `B=(7,3)` are collections of 3D vectors. Compute every dot product to produce `C=(10,7)`:

$$C_{ij}=\sum_k A_{ik} B_{jk}$$

Implement it **three ways**: (1) broadcasting + `.sum`, (2) broadcasting + `einops.reduce`, (3) matrix multiplication. Verify with `t.allclose`.

In [ ]:
A = t.randn(10,3); B = t.randn(7,3)
# YOUR WORK
# C_broadcast = ...
# C_einops = ...
# C_matmul = ...
# assert C_broadcast.shape == (10,7)
# assert t.allclose(C_broadcast, C_einops)
# assert t.allclose(C_broadcast, C_matmul, atol=1e-5)

**Solution 6**

```python
products = A[:,None,:] * B[None,:,:]       # (10,7,3)
C_broadcast = products.sum(dim=-1)          # (10,7)
C_einops = reduce(products, "a b d -> a b", "sum")
C_matmul = A @ B.T
assert t.allclose(C_broadcast, C_einops)
assert t.allclose(C_broadcast, C_matmul, atol=1e-5)
```

## Bonus · Einsum (optional)

Repeated labels not in the output are summed over:

```python
C = t.einsum("ik,jk->ij", A, B)
```

For attention scores, `(batch, heads, query, d)` and `(batch, heads, key, d)` contract over `d` to yield `(batch, heads, query, key)`.

In [ ]:
Q = t.randn(2,4,10,64); K = t.randn(2,4,10,64)
scores = t.einsum("b h q d, b h k d -> b h q k",Q,K)
print(scores.shape)

## Cheat sheet & ARENA transfer

| Task | PyTorch | Einops |
|---|---|---|
| Swap axes | `x.transpose(1,2)` | `rearrange(x, "b s d -> b d s")` |
| Flatten | `x.reshape(b,-1)` | `rearrange(x, "b s d -> b (s d)")` |
| Add singleton axis | `x[:,None,:]` | `rearrange(x, "b d -> b 1 d")` |
| Add repeated axis | `x[:,None,:,:].repeat(1,4,1,1)` | `repeat(x, "b s d -> b n s d", n=4)` |
| Mean across tokens | `x.mean(dim=1)` | `reduce(x, "b s d -> b d", "mean")` |
| Matrix multiply | `A @ B` | — |

**Return to ARENA:** before coding the ray–segment intersection, predict each shape:

```text
Ray directions:       (nrays, 2)
Segment directions:   (nsegments, 2)
Pairwise matrices:    (nrays, nsegments, 2, 2)
Right-hand sides:     (nrays, nsegments, 2)
Solutions:            (nrays, nsegments, 2)
Pairwise intersections:(nrays, nsegments)
Final result:         (nrays,)
```

**Self-check:** Can you explain which axis is added, broadcast, contracted, or reduced at each step?

References: [PyTorch broadcasting](https://docs.pytorch.org/docs/stable/notes/broadcasting.html) · [Einops basics](https://einops.rocks/1-einops-basics/)